In [4]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Embedding, Dense,LSTM,GRU
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [5]:
vocab_size= 10000
max_len= 200
embedding_dim= 128

#load data
(x_train,y_train),(x_test,y_test)= imdb.load_data(num_words=vocab_size)

#pad_sequence
x_train= pad_sequences(x_train, maxlen=max_len)
x_test= pad_sequences(x_test, maxlen=max_len)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [6]:
#model- Vanilla RNN

model= Sequential([
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len),
    SimpleRNN(64),
    Dense(1,activation='sigmoid')])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
#model- Stacked RNN

model= Sequential([
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len),
    SimpleRNN(64, return_sequences=True),
    SimpleRNN(32),
    Dense(1,activation='sigmoid')])

In [9]:
#model- Vanilla LSTM

model= Sequential([
    Embedding(vocab_size,embedding_dim,input_length= max_len),
    LSTM(64),
    Dense(1,activation='sigmoid')])



In [10]:
#model- Stacked LSTM

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_len),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(1, activation="sigmoid")
])

In [11]:
#model- GRU

model= Sequential([
    Embedding(vocab_size,embedding_dim,input_length=max_len),
    GRU(64),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:
#compile
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [13]:
#Train

model.fit(x_train,y_train,epochs=5,batch_size=64,validation_split=0.2)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 97s 300ms/step - accuracy: 0.7632 - loss: 0.4699 - val_accuracy: 0.8324 - val_loss: 0.3791
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 93s 298ms/step - accuracy: 0.8881 - loss: 0.2746 - val_accuracy: 0.8706 - val_loss: 0.3170
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 94s 301ms/step - accuracy: 0.9193 - loss: 0.2042 - val_accuracy: 0.8586 - val_loss: 0.3388
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 101s 324ms/step - accuracy: 0.9528 - loss: 0.1284 - val_accuracy: 0.8666 - val_loss: 0.3800
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 133s 295ms/step - accuracy: 0.9714 - loss: 0.0851 - val_accuracy: 0.8504 - val_loss: 0.4039


In [14]:
#Evaluate
test_loss, test_acc= model.evaluate(x_test,y_test)
print(f"Test accuracy",test_acc)

782/782 ━━━━━━━━━━━━━━━━━━━━ 27s 34ms/step - accuracy: 0.8503 - loss: 0.4210
Test accuracy 0.8502799868583679


In [15]:
#Prediction:
from tensorflow.keras.datasets import imdb
import numpy as np

word_index= imdb.get_word_index()
reverse_word_index= {v+3:k for k,v in word_index.items()}
reverse_word_index[0] ="<PAD>"
reverse_word_index[1] ="<Start>"
reverse_word_index[2] ="<UNK>"

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [16]:
sample_id = 8

sample_review = x_test[sample_id]
true_label = y_test[sample_id]

In [17]:
decoded_review = " ".join(
    reverse_word_index.get(i, "?") for i in sample_review if i != 0
)

print("Review Text:\n")
print(decoded_review[:1000])  # print first 1000 chars
print("\nTrue Label:", "Positive" if true_label == 1 else "Negative")

Review Text:

<Start> hollywood had a long love affair with bogus <UNK> nights tales but few of these products have stood the test of time the most memorable were the jon hall maria <UNK> films which have long since become camp this one is filled with dubbed songs <UNK> <UNK> and slapstick it's a truly crop of corn and pretty near <UNK> today it was nominated for its imaginative special effects which are almost <UNK> in this day and age <UNK> mainly of trick photography the only outstanding positive feature which survives is its beautiful color and clarity sad to say of the many films made in this genre few of them come up to alexander <UNK> original thief of <UNK> almost any other <UNK> nights film is superior to this one though it's a loser

True Label: Negative


In [18]:
# Model expects batch dimension
sample_input = np.expand_dims(sample_review, axis=0)

prediction = model.predict(sample_input)[0][0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step


In [19]:
sentiment = "Positive" if prediction >= 0.5 else "Negative"
confidence = prediction if prediction >= 0.5 else 1 - prediction

print("\nModel Prediction:")
print("Sentiment:", sentiment)
print("Confidence:", round(confidence * 100, 2), "%")


Model Prediction:
Sentiment: Positive
Confidence: 97.12 %
